# **Sydney T1 Duty-Free Shop — Synthetic Dataset**

> #### ⚠️ **Disclaimer**  
> This project is based on personal retail work experience and uses a **fully synthetic dataset built around a hypothetical scenario**.  All data is artificially generated — no real customer, transaction, or operational data has been used or reproduced.  
> *"This dataset was designed from first-hand retail experience to reflect real-world airport duty-free dynamics as accurately as possible."*

A portfolio dataset simulating the sales operations of a **duty-free gift shop at Sydney International Airport Terminal 1 (T1)**.  
- Designed to practise SQL analytics, data modelling, and Tableau visualisation — covering realistic retail patterns such as flight-linked transactions, nationality-based purchasing behaviour, and multi-type promotions.

| Item | Detail |
|---|---|
| **Setting** | Sydney International Airport — Terminal 1 duty-free gift shop |
| **Period** | 1 January 2024 – 30 June 2024 (26 weeks) |
| **Tables** | 6 |
| **Total records** | ~20,000+ rows |
| **Output** | 5 CSV files + 1 SQLite database |
| **Reproducibility** | `random.seed(42)` fixed |

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os
import sqlite3

In [2]:
fake = Faker()
random.seed(42)
np.random.seed(42)

In [28]:
OUTPUT_DIR = "duty_free_data"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"✨ '{OUTPUT_DIR}' created.")

os.makedirs(OUTPUT_DIR, exist_ok=True)

✨ 'duty_free_data' created.


## 1. COMMON CONSTANTS


- Nationality weights based on Sydney Airport Q4 2024 official traffic report

- Rank (traffic): AU(1) > CN(2) > NZ(3) > US(4) > GB(5) > KR(6) > IN(7) > JP(8) > PH(9) > CA(10)

- AU weight (12%) is lower than raw traffic share (22%) but accounts for:
    
    ① Naturalised AU citizens travelling back to their hometown (CN/KR/etc origin)

    — these travellers have similar purchase behaviour to their origin nationality

    ② Other AU travellers have low duty-free conversion (buy locally)

- Note: NZ weight (8%) kept lower than raw traffic share:

    ① NZ biosecurity law restricts honey & some food imports from AU (MPI regulations)

    ② Many NZ-AU travellers are residents who can buy the same products locally

In [4]:
NATIONALITIES      = ["AU", "CN", "NZ", "US", "GB", "KR", "IN", "JP", "PH", "CA", "SG", "TH"]
NATIONALITY_WEIGHT = [0.12, 0.20, 0.08, 0.10, 0.09, 0.09, 0.08, 0.07, 0.05, 0.04, 0.04, 0.04]

MEMBERSHIP_LEVELS  = ["Non-Member", "Silver", "Gold", "Diamond"]
MEMBERSHIP_WEIGHT  = [0.55, 0.25, 0.13, 0.07]

AGE_GROUPS = ["20s", "30s", "40s", "50s+"]

CATEGORIES = [
    "Cosmetics", "Liquor", "Jewellery", "Souvenir",
    "Confectionery", "Apparel", "Honey", "Indigenous", "Tea"
]

PAYMENT_METHODS = [
    "Credit/Debit Card",  # physical card tap / swipe
    "Cash",               # notes and coins
    "AliPay",             # CN digital wallet (QR scan)
    "WeChat Pay",         # CN digital wallet (QR scan)
    "Digital Wallet",     # Apple Pay / Google Pay (NFC)
]

## 2. FLIGHT DEFINITIONS


Each entry : (flight_no, destination, via, airline, dep_hour, dep_min, weekdays)

- Flight numbers are fixed in real-world format (CX101, KE601 ...)
- Operating weekdays fixed per route  (0=Mon ~ 6=Sun)
- Northeast Asia  : morning to afternoon departures
- Southeast Asia  : morning departures
- Long-haul (Dubai/Doha/LA) : night departures

In [5]:
FLIGHT_DEF = [
    # ── Northeast Asia ──────────────────────────────────────
    ("CX101", "Hong Kong",    "Direct", "Cathay Pacific",        10, 30, [0,1,2,3,4,5,6]),  # daily
    ("CX103", "Hong Kong",    "Direct", "Cathay Pacific",        15, 45, [0,2,4,6]),         # Mon/Wed/Fri/Sun
    ("MU502", "Shanghai",     "Direct", "China Eastern",          9, 15, [0,1,3,4,6]),       # 5x/week
    ("CA781", "Beijing",      "Direct", "Air China",             10, 0,  [1,3,5]),            # 3x/week
    ("KE601", "Seoul",        "Direct", "Korean Air",            11, 30, [0,1,2,3,4,5,6]),  # daily
    ("OZ602", "Seoul",        "Direct", "Asiana Airlines",        9, 45, [0,2,4,5]),         # 4x/week
    ("JL771", "Tokyo",        "Direct", "Japan Airlines",        10, 0,  [0,1,2,3,4,5,6]),  # daily
    ("NH880", "Tokyo",        "Direct", "ANA",                   11, 15, [1,3,5,6]),         # 4x/week

    # ── Southeast Asia ──────────────────────────────────────
    ("SQ211", "Singapore",    "Direct", "Singapore Airlines",     8, 0,  [0,1,2,3,4,5,6]),  # daily
    ("SQ213", "Singapore",    "Direct", "Singapore Airlines",    13, 30, [0,2,3,5,6]),       # 5x/week
    ("TR13",  "Singapore",    "Direct", "Scoot",                  7, 45, [1,3,5]),            # 3x/week
    ("PR211", "Manila",       "Direct", "Philippine Airlines",    9, 30, [0,2,4,5,6]),       # 5x/week
    ("5J32",  "Manila",       "Direct", "Cebu Pacific",           8, 0,  [1,4,6]),            # 3x/week
    ("MH122", "Kuala Lumpur", "Direct", "Malaysia Airlines",      8, 45, [0,1,3,4,6]),       # 5x/week
    ("TG476", "Bangkok",      "Direct", "Thai Airways",           9, 0,  [0,2,3,5,6]),       # 5x/week
    ("GA715", "Bali",         "Direct", "Garuda Indonesia",       7, 30, [0,1,2,3,4,5,6]),  # daily

    # ── Long-haul (night departures) ────────────────────────
    ("QF1",   "Singapore",    "Direct", "Qantas",                16, 30, [0,1,2,3,4,5,6]),  # daily SYD→SIN (connects SIN→LHR as QF2)
    ("EK413", "Dubai",        "Direct", "Emirates",              21, 10, [0,1,2,3,4,5,6]),  # daily SYD→DXB
    ("QR007", "Doha",         "Direct", "Qatar Airways",         20, 45, [0,2,4,6]),          # 4x/week SYD→DOH
    ("QF7",   "Los Angeles",  "Direct", "Qantas",                11, 55, [0,1,2,3,4,5,6]),  # daily
    ("UA839", "Los Angeles",  "Direct", "United Airlines",       21, 0,  [1,3,5,6]),         # 4x/week
]

## 3. ITEMS / SKU VARIANT DEFINITIONS / COST RATIOS


### 3.1 ITEMS

In [6]:
ITEM_MAP = {
    # ── Cosmetics ─────────────────────────────────────────
    # Item names used for PREF_MAP lookups; variants defined in COSMETICS_SKU_VARIANTS
    "Cosmetics": [
        "Lanolin Cream",
        "Sheep Placenta Cream",
        "Emu Oil Cream",
        "Paw Paw Ointment",
        "Wild Fern Skincare",
    ],
    # ── Liquor ────────────────────────────────────────────
    # Item names used for PREF_MAP lookups; variants defined in LIQUOR_SKU_VARIANTS
    "Liquor": [
        "Penfolds Wine",
        "Johnnie Walker Whisky",
        "Hennessy Cognac",
        "Absolut Vodka",
        "Tanqueray Gin",
    ],
    # ── Jewellery ─────────────────────────────────────────
    # AU/NZ specialty jewellery — opal, paua shell & jade
    "Jewellery": [
        "Opal Jewellery",        # AU opal — wide price range by grade
        "Paua Shell Jewellery",  # NZ paua shell — iridescent blue/green
        "Jade Jewellery",        # Australian/International/New Zealand jade 
    ],
    # ── Souvenir ──────────────────────────────────────────
    # Sydney / Australia themed gift items
    "Souvenir": [
        "SYD Keychain",
        "SYD Mug",
        "AUS Magnet",
        "Kangaroo Plush",
        "Koala Plush",
        "AUS Tea Towel",
    ],
    # ── Confectionery ─────────────────────────────────────
    # Branded chocolate + AU specialty snacks
    "Confectionery": [
        "Cadbury Chocolate",  # global chocolate (AU-produced)
        "Tim Tam Biscuit",    # AU iconic biscuit
        "Patons Chocolate",   # AU chocolate
        "Lindt Chocolate",    # global premium
        "Macadamia Nuts",     # AU specialty nut
        "Kangaroo Jerky",     # AU airport snack
    ],
    # ── Apparel ───────────────────────────────────────────
    # AU themed clothing & specialty items
    "Apparel": [
        "AUS Souvenir T-Shirt",
        "AUS Souvenir Hoodie",
        "Ugg Boots",
        "Ugg Slipper",
        "AUS Souvenir Cap",
    ],
    # ── Honey ─────────────────────────────────────────────
    # Brand names only — grade & price assigned per SKU in HONEY_SKU_VARIANTS
    # Each brand generates multiple SKUs (one per grade)
    "Honey": [
        "Comvita",       # 4 grades: UMF 5+ / 10+ / 15+ / 20+
        "Manuka Health", # 2 grades: MGO 263+ / 400+
        "Beepower",      # 2 grades: MGO 115+ / 263+
    ],
    # ── Indigenous ────────────────────────────────────────
    # Aboriginal art & AU native beauty products
    "Indigenous": [
        "Aboriginal Art Print",   # aboriginal art print
        "Boomerang",              # traditional boomerang
        "Indigenous Hand Cream",  # native hand cream
        "Indigenous Lip Balm",    # native lip balm
        "Indigenous Kitchenware", # aboriginal pattern kitchenware
    ],
    # ── Tea ───────────────────────────────────────────────
    # AU tea brands
    "Tea": [
        "T2 Tea"
    ]
}

### 3.2 SKU VARIANT DEFINITIONS


- All categories use variant-based SKU generation.
- Format: (Item, Variant, Price AUD)


In [7]:
# ── Cosmetics ─────────────────────────────────────────────────
COSMETICS_SKU_VARIANTS = [
    # Lanolin Cream
    ("Lanolin Cream", "A Brand 100ml",           8),
    ("Lanolin Cream", "B Brand 100ml",          12),
    ("Lanolin Cream", "A Brand 6-pack 100ml",   36),
    ("Lanolin Cream", "B Brand 6-pack 100ml",   60),
    # Sheep Placenta Cream
    ("Sheep Placenta Cream", "A Brand 100ml",         9),
    ("Sheep Placenta Cream", "B Brand 100ml Premium",22),
    ("Sheep Placenta Cream", "A Brand 6-pack 100ml", 45),
    # Emu Oil Cream
    ("Emu Oil Cream", "A Brand 100ml",           9),
    ("Emu Oil Cream", "A Brand 6-pack 100ml",   45),
    # Paw Paw Ointment
    ("Paw Paw Ointment", "L Paw Paw 75g",     10),
    ("Paw Paw Ointment", "G Paw Paw 75g",      9),
    # Wild Fern Skincare
    ("Wild Fern Skincare", "Lip Balm",          12),
    ("Wild Fern Skincare", "Hand Cream",        18),
    ("Wild Fern Skincare", "Face Cream",        27),
    ("Wild Fern Skincare", "Face Mask",         11),
    ("Wild Fern Skincare", "Gift Set",          42),
]

# ── Liquor ────────────────────────────────────────────────────
LIQUOR_SKU_VARIANTS = [
    # Penfolds Wine
    ("Penfolds Wine", "Max's",     30),
    ("Penfolds Wine", "Bin 407",  115),
    ("Penfolds Wine", "Bin 28",    35),
    ("Penfolds Wine", "Bin 389",  120),
    ("Penfolds Wine", "Grange",   999),
    # Johnnie Walker Whisky
    ("Johnnie Walker Whisky", "Red Label 700ml",   40),
    ("Johnnie Walker Whisky", "Black Label 700ml",  55),
    ("Johnnie Walker Whisky", "Gold Label 700ml",  100),
    # Hennessy Cognac
    ("Hennessy Cognac", "VS 700ml",    62),
    ("Hennessy Cognac", "VSOP 700ml",  88),
    ("Hennessy Cognac", "XO 700ml",   210),
    # Absolut Vodka
    ("Absolut Vodka", "Original 700ml",    42),
    ("Absolut Vodka", "Flavoured 700ml",   45),
    ("Absolut Vodka", "Elyx 700ml",        72),
    # Tanqueray Gin
    ("Tanqueray Gin", "London Dry 700ml",  48),
    ("Tanqueray Gin", "No. Ten 700ml",     65),
    ("Tanqueray Gin", "Rangpur 700ml",     52),
]

# ── Jewellery ─────────────────────────────────────────────────
JEWELLERY_SKU_VARIANTS = [
    # Opal Jewellery
    ("Opal Jewellery",       "Earrings",         85),
    ("Opal Jewellery",       "Necklace",        120),
    ("Opal Jewellery",       "Pendant",         220),
    ("Opal Jewellery",       "Premium Opal",    520),
    # Paua Shell Jewellery
    ("Paua Shell Jewellery", "Earrings",         38),
    ("Paua Shell Jewellery", "Pendant",          55),
    ("Paua Shell Jewellery", "Bracelet",         75),
    ("Paua Shell Jewellery", "Ring",             90),
    # Jade Jewellery
    ("Jade Jewellery",       "Pendant Small",    65),
    ("Jade Jewellery",       "Pendant Large",   110),
    ("Jade Jewellery",       "Bracelet",        150),
    ("Jade Jewellery",       "Ring",            200),
]

# ── Souvenir ──────────────────────────────────────────────────
SOUVENIR_SKU_VARIANTS = [
    # SYD Keychain
    ("SYD Keychain",   "Metal",          12),
    ("SYD Keychain",   "Acrylic",         9),
    # SYD Mug
    ("SYD Mug",        "Standard 350ml", 20),
    ("SYD Mug",        "Large 500ml",    25),
    # AUS Magnet
    ("AUS Magnet",     "Standard",       10),
    ("AUS Magnet",     "3-Pack",         18),
    # Kangaroo Plush
    ("Kangaroo Plush", "Small 20cm",     18),
    ("Kangaroo Plush", "Large 40cm",     45),
    # Koala Plush
    ("Koala Plush",    "Small 20cm",     18),
    ("Koala Plush",    "Large 40cm",     45),
    # AUS Tea Towel
    ("AUS Tea Towel",  "Single",         18),
]

# ── Confectionery ─────────────────────────────────────────────
CONFECTIONERY_SKU_VARIANTS = [
    # Cadbury Chocolate (3 SKUs)
    ("Cadbury Chocolate", "Milk Chocolate 200g",   9),
    ("Cadbury Chocolate", "Gift Box Assorted",     16),
    ("Cadbury Chocolate", "Favourites Gift Box",   22),
    # Tim Tam — flavour variants
    ("Tim Tam Biscuit", "Original",                7),
    ("Tim Tam Biscuit", "Dark Chocolate",          7),
    ("Tim Tam Biscuit", "White",                   7),
    ("Tim Tam Biscuit", "Double Coat",             7),
    ("Tim Tam Biscuit", "Gluten Free",             7),
    ("Tim Tam Biscuit", "Chewy Caramel",           7),
    ("Tim Tam Biscuit", "Jatz",                    7),
    # Patons Chocolate (3 SKUs)
    ("Patons Chocolate", "Milk Chocolate 200g",   22),
    ("Patons Chocolate", "Dark Chocolate 200g",   22),
    ("Patons Chocolate", "Gift Box",              27),
    # Lindt Chocolate (3 SKUs)
    ("Lindt Chocolate", "Excellence 70% 100g",    12),
    ("Lindt Chocolate", "Lindor Box 200g",        22),
    ("Lindt Chocolate", "Gift Box Assorted",      35),
    # Macadamia Nuts (3 SKUs)
    ("Macadamia Nuts", "Honey Roasted 100g",      13),
    ("Macadamia Nuts", "Mixed Flavour 200g",      27),
    ("Macadamia Nuts", "Gift Tin 300g",           36),
    # Kangaroo Jerky — weight variants
    ("Kangaroo Jerky", "Original 50g",            11),
    ("Kangaroo Jerky", "Original 100g",           17),
]

# ── Apparel ───────────────────────────────────────────────────
APPAREL_SKU_VARIANTS = [
    # AUS Souvenir T-Shirt
    ("AUS Souvenir T-Shirt", "Kids",      28),
    ("AUS Souvenir T-Shirt", "Adult",     40),
    # AUS Souvenir Hoodie
    ("AUS Souvenir Hoodie",  "Standard",  55),
    ("AUS Souvenir Hoodie",  "Premium",   80),
    # Ugg Boots
    ("Ugg Boots",            "Classic Short", 110),
    ("Ugg Boots",            "Classic Tall",  180),
    # Ugg Slipper
    ("Ugg Slipper",          "Slipper",       100),
    # AUS Souvenir Cap
    ("AUS Souvenir Cap",     "Standard",   18),
    ("AUS Souvenir Cap",     "Premium",    27),
]

# ── Honey ─────────────────────────────────────────────────────
HONEY_SKU_VARIANTS = [
    ("Comvita",       "UMF 5+",    45),
    ("Comvita",       "UMF 10+",   60),
    ("Comvita",       "UMF 15+",   90),
    ("Comvita",       "UMF 20+",  170),
    ("Manuka Health", "MGO 263+",  55),
    ("Manuka Health", "MGO 400+",  75),
    ("Beepower",      "MGO 115+",  35),
    ("Beepower",      "MGO 263+",  45),
]

# ── Indigenous ────────────────────────────────────────────────
INDIGENOUS_SKU_VARIANTS = [
    # Aboriginal Art Print
    ("Aboriginal Art Print",   "A4 Print",       28),
    ("Aboriginal Art Print",   "A3 Print",       55),
    ("Aboriginal Art Print",   "Framed A3",     110),
    # Boomerang
    ("Boomerang",              "Painted Small",  25),
    ("Boomerang",              "Painted Large",  55),
    # Indigenous Hand Cream
    ("Indigenous Hand Cream",  "50ml",           14),
    ("Indigenous Hand Cream",  "100ml",          24),
    # Indigenous Lip Balm
    ("Indigenous Lip Balm",    "Single",          9),
    ("Indigenous Lip Balm",    "3-Pack",         18),
    # Indigenous Kitchenware
    ("Indigenous Kitchenware", "Coaster Set",    22),
    ("Indigenous Kitchenware", "Cutting Board",  48),
]

# ── Tea ───────────────────────────────────────────────────────
# T2 only — Dilmah / Twinings / Madura removed per design decision
TEA_SKU_VARIANTS = [
    ("T2 Tea", "Small Tin 50g",  22),
    ("T2 Tea", "Large Tin 100g", 42),
    ("T2 Tea", "Gift Set",       62),
]

CN customers show strong preference for high-value items:

① Honey : UMF 10+ / MGO 400+ and above (gift-quality grades)

② Liquor: $85+ bottles (Hennessy VSOP/XO, JW Gold, Penfolds Bin 407/389/Grange)

In [8]:
# ── CN nationality-based SKU preference ──────────────────────
CN_PREMIUM_HONEY_VARIANTS   = ["UMF 10+", "UMF 15+", "UMF 20+", "MGO 400+"]
CN_PREMIUM_LIQUOR_MIN_PRICE = 85.0
CN_PREMIUM_TRIGGER_RATE     = 0.70

In [9]:
# PRICE_RANGE — fallback only (not reached in normal execution)
# Reflects actual min/max across all variants in each category's SKU_VARIANTS list
PRICE_RANGE = {
    "Cosmetics"    : (8,    42),  # A Brand 100ml Lanolin Cream $8 → Wild Fern Gift Set $42
    "Liquor"       : (30,  999),  # Penfolds Max's $30 → Grange $999
    "Jewellery"    : (38,  520),  # Paua Earrings $38 → Opal Premium $520
    "Souvenir"     : (9,    45),  # AUS Magnet Acrylic $9 → Kangaroo Plush Large $45
    "Confectionery": (7,    36),  # Tim Tam $7 → Macadamia Gift Tin $36
    "Apparel"      : (22,  180),  # AUS Cap Standard $22 → Ugg Classic Tall $180
    "Honey"        : (35,  170),  # Beepower MGO 115+ $35 → Comvita UMF 20+ $170
    "Indigenous"   : (9,   110),  # Indigenous Lip Balm Single $9 → Aboriginal Art Framed A3 $110
    "Tea"          : (22,   62),  # T2 Small Tin $22 → T2 Gift Set $62
}

### 3.3 Item-level promotions

- These run year-round (not tied to holidays)
- Format: (promo_id, description, category, item_filter, promo_type, params)

- Promo types used:
    - BUY_X_GET_Y_ITEM  : buy X of matching item, get 1 free (same item)
    - BUNDLE_FIXED_ITEM : N of matching item for fixed price
    - MIX_BUY_X_GET_Y   : buy X from category pool, get 1 free (random SKU, paid qty only)

In [10]:
ITEM_PROMOS = [
    # Cosmetics — Paw Paw Ointment: Buy 5 Get 6th Free
    # G Paw Paw ($9) and L Paw Paw ($10) both eligible — mix allowed
    # The 6th (free) item is always G Paw Paw ($9) — cheapest of the two
    # free_item_price : retail price of the free item (excluded from Total_Amount)
    # free_item_cost  : cost of the free item (included in Total_Cost for margin accuracy)
    {
        "promo_id"        : "IP-01",
        "description"     : "Paw Paw Ointment Buy 5 Get 6th Free (G Paw Paw free)",
        "category"        : "Cosmetics",
        "item_filter"     : "Paw Paw Ointment",
        "variant_filter"  : None,               # both G and L eligible to buy
        "promo_type"      : "MIX_BUY_X_GET_Y",
        "buy_qty"         : 5,
        "free_qty"        : 1,
        "bundle_price"    : None,
        "free_item_price" : 9.0,                # G Paw Paw retail price
        "free_item_cost"  : round(9.0 * 0.40, 2),  # G Paw Paw cost ($3.60)
    },
    # Honey — Comvita UMF 5+ only: Buy 3 Get 4th Free
    {
        "promo_id"   : "IP-02",
        "description": "Comvita UMF 5+ Buy 3 Get 4th Free",
        "category"   : "Honey",
        "item_filter": "Comvita",             # Item == "Comvita", Variant == "UMF 5+"
        "variant_filter": "UMF 5+",
        "promo_type" : "BUY_X_GET_Y_ITEM",
        "buy_qty"    : 3,
        "free_qty"   : 1,
        "bundle_price": None,
    },
    # Confectionery — Tim Tam any flavour: 4 for $24
    # All Tim Tam SKUs are $7 → 4×$7=$28 → bundle $24 saves $4
    {
        "promo_id"   : "IP-03",
        "description": "Tim Tam Any Flavour 4 for $24",
        "category"   : "Confectionery",
        "item_filter": "Tim Tam Biscuit",
        "variant_filter": None,               # any Tim Tam flavour
        "promo_type" : "BUNDLE_FIXED_ITEM",
        "buy_qty"    : 4,
        "free_qty"   : 0,
        "bundle_price": 24.0,
    },
]

In [11]:
# Probability that an eligible transaction triggers a promo
# (not every customer buying that item will take the promo deal)
PROMO_TRIGGER_RATE = {
    "IP-01": 0.20,   # 20% of Paw Paw buyers take the 6-pack deal
    "IP-02": 0.25,   # 25% of Comvita UMF 5+ buyers buy 4
    "IP-03": 0.40,   # 30% of Tim Tam buyers take the 4-pack deal
}

### 3.4 Cost Ratio

Cost_Price = Selling_Price × COST_RATIO

In [12]:
# ── Cost ratios ───────────────────────────────────────────────
COST_RATIO = {
    "Cosmetics"    : 0.40,
    "Liquor"       : 0.55,
    "Jewellery"    : 0.40,
    "Souvenir"     : 0.35,
    "Confectionery": 0.50,
    "Apparel"      : 0.40,
    "Honey"        : 0.45,
    "Indigenous"   : 0.30,
    "Tea"          : 0.40,
}

## 4. NATIONALITY × PREFERRED CATEGORY  (weighted tuples)


- Weights do NOT need to sum to 100.
- pick_category() distributes any remainder evenly across
- unlisted categories, so every category has some purchase probability.
- Listed categories = strong preferences; remainder = background noise.

In [13]:
PREF_MAP = {
    # AU: lower duty-free conversion than raw traffic share
    # duty-free price benefit (main driver)
    # snacks/gifts for family & friends
    # Paw Paw, Lanolin — pharmacy substitute at airport
    "AU": [("Liquor",15),("Confectionery",25),("Cosmetics",10),("Souvenir",10)],

    # CN: manuka honey + cosmetics + cognac + souvenirs (gift-buying culture)
    "CN": [("Honey",30),("Cosmetics",26),("Liquor",16),("Confectionery",12),("Souvenir",16)],

    # NZ: souvenirs + confectionery + tea
    # Honey very low — NZ MPI biosecurity restricts AU honey imports
    "NZ": [("Souvenir",30),("Confectionery",25),("Tea",20),("Cosmetics",10),("Honey",5)],

    # US: apparel + indigenous + souvenir + jewellery
    # Liquor reduced — not a strong duty-free driver for US travellers at AU airports
    "US": [("Apparel",28),("Indigenous",24),("Souvenir",20),("Jewellery",12)],

    # GB: tea + apparel + indigenous
    # Liquor reduced — not a strong duty-free driver for GB travellers at AU airports
    "GB": [("Tea",30),("Apparel",24),("Indigenous",18),("Souvenir",12)],

    # KR: cosmetics + honey + wine (occasional) + confectionery
    # Honey and confectionery bought at similar rates
    "KR": [("Cosmetics",35),("Honey",22),("Confectionery",20),("Liquor",10),("Jewellery",8)],

    # IN: confectionery + souvenir + cosmetics
    "IN": [("Confectionery",32),("Souvenir",25),("Tea",18),("Cosmetics",10)],

    # JP: confectionery + souvenir + tea (omiyage gift culture)
    "JP": [("Confectionery",38),("Souvenir",28),("Tea",18),("Cosmetics",8)],

    # PH: confectionery + souvenir + cosmetics
    "PH": [("Confectionery",35),("Souvenir",30),("Cosmetics",18),("Tea",8)],

    # CA: apparel + indigenous + souvenir
    # Liquor reduced — similar to US/GB pattern
    "CA": [("Apparel",28),("Indigenous",24),("Souvenir",20),("Tea",10)],

    # SG: cosmetics + confectionery + honey
    "SG": [("Cosmetics",30),("Confectionery",24),("Honey",18),("Tea",12),("Souvenir",8)],

    # TH: cosmetics + souvenir + confectionery
    "TH": [("Cosmetics",32),("Souvenir",26),("Confectionery",20),("Tea",8)],
}

In [14]:
DEST_CAT_BIAS = {
    # Northeast Asia routes: Honey & Cosmetics are strong sellers (CN/KR/JP travellers)
    "asia_east": ["Cosmetics","Honey","Confectionery","Tea"],
    # Southeast Asia routes: Souvenir & Confectionery dominant
    "asia_se"  : ["Souvenir","Confectionery","Cosmetics","Tea"],
    # Long-haul routes: Liquor, Apparel, Jewellery, Indigenous art
    "long_haul": ["Liquor","Apparel","Jewellery","Indigenous"],
}

In [15]:
DEST_GROUP = {
    # Northest Asia
    "Hong Kong":"asia_east",
    "Shanghai":"asia_east",
    "Beijing":"asia_east",
    "Seoul":"asia_east",
    "Tokyo":"asia_east",
    # Southeast Asia
    "Singapore":"asia_se",
    "Manila":"asia_se",
    "Kuala Lumpur":"asia_se",
    "Bangkok":"asia_se",
    "Bali":"asia_se",
    # Long-haul
    "Dubai":"long_haul",
    "Doha":"long_haul",
    "Los Angeles":"long_haul",
}

In [16]:
def pick_category(nat: str, dest: str, active_holiday: dict = None) -> str:
    """
    Resolve purchase category using two-stage weighted selection.

    Stage 1 (70% probability) — nationality preference:
      Uses PREF_MAP weights. If total < 100, the remainder is
      distributed evenly across unlisted categories so every
      category has some non-zero purchase probability.
      If a holiday is active, the target category receives an
      additional Boost_Weight on top of its normal weight.

    Stage 2 (30% probability) — destination group bias:
      Uses DEST_CAT_BIAS for the flight's destination group.
    """
    if random.random() < 0.70:
        pref_opts = PREF_MAP.get(nat, [])
        listed_cats   = {c for c, _ in pref_opts}
        listed_weight = sum(w for _, w in pref_opts)
        remainder     = max(0, 100 - listed_weight)

        # Distribute remainder evenly across unlisted categories
        unlisted_cats = [c for c in CATEGORIES if c not in listed_cats]
        base_weight   = remainder / len(unlisted_cats) if unlisted_cats else 0

        all_cats    = [c for c, _ in pref_opts] + unlisted_cats
        all_weights = [w for _, w in pref_opts] + [base_weight] * len(unlisted_cats)

        # Apply holiday boost — add extra weight to target category
        if active_holiday:
            boost_cat = active_holiday["Cat"]
            boost_val = active_holiday["Boost"]
            if boost_cat in all_cats:
                idx = all_cats.index(boost_cat)
                all_weights[idx] += boost_val
            else:
                all_cats.append(boost_cat)
                all_weights.append(boost_val)

        return random.choices(all_cats, weights=all_weights)[0]
    else:
        group = DEST_GROUP.get(dest)
        pool  = DEST_CAT_BIAS.get(group, CATEGORIES) if group else CATEGORIES
        return random.choice(pool)


## 5. Table 1 — Customer_Profiles  (500 customers)


In [17]:
def generate_customers(n: int = 500) -> pd.DataFrame:
    records = []
    for i in range(1, n + 1):
        nat = random.choices(NATIONALITIES, weights=NATIONALITY_WEIGHT)[0]
        opts = PREF_MAP.get(nat, [(c,1) for c in CATEGORIES])
        pref = random.choices([o[0] for o in opts], weights=[o[1] for o in opts])[0]
        records.append({
            "Customer_ID"       : f"C-{i:04d}",
            "Nationality"       : nat,
            "Membership_Level"  : random.choices(MEMBERSHIP_LEVELS, weights=MEMBERSHIP_WEIGHT)[0],
            "Age_Group"         : random.choice(AGE_GROUPS),
            "Registration_Date" : fake.date_between(start_date="-3y", end_date="-6m"),
            "Preferred_Category": pref,
        })
    return pd.DataFrame(records)

## 6. Table 2 — Product_Inventory  (10 SKUs per category, 90 total)


In [18]:
def generate_products() -> pd.DataFrame:
    """
    Build Product_Inventory — one row per SKU variant tuple.
    All categories use variant-based generation (no range-based random).

    Category      Variant list                  SKUs
    ──────────    ──────────────────────────    ────
    Cosmetics   → COSMETICS_SKU_VARIANTS         16
    Liquor      → LIQUOR_SKU_VARIANTS            17
    Jewellery   → JEWELLERY_SKU_VARIANTS         12
    Souvenir    → SOUVENIR_SKU_VARIANTS          11
    Confectionery→ CONFECTIONERY_SKU_VARIANTS    21
    Apparel     → APPAREL_SKU_VARIANTS            9
    Honey       → HONEY_SKU_VARIANTS              8
    Indigenous  → INDIGENOUS_SKU_VARIANTS        11
    Tea         → TEA_SKU_VARIANTS                3
    ──────────────────────────────────────────────
    Total                                       108
    """
    records, sku_n = [], 1

    VARIANT_MAP = {
        "Cosmetics"    : COSMETICS_SKU_VARIANTS,
        "Liquor"       : LIQUOR_SKU_VARIANTS,
        "Jewellery"    : JEWELLERY_SKU_VARIANTS,
        "Souvenir"     : SOUVENIR_SKU_VARIANTS,
        "Confectionery": CONFECTIONERY_SKU_VARIANTS,
        "Apparel"      : APPAREL_SKU_VARIANTS,
        "Honey"        : HONEY_SKU_VARIANTS,
        "Indigenous"   : INDIGENOUS_SKU_VARIANTS,
        "Tea"          : TEA_SKU_VARIANTS,
    }

    for cat in CATEGORIES:
        for (item, variant, price) in VARIANT_MAP[cat]:
            safety = random.randint(20, 50)
            records.append({
                "Product_SKU"  : f"SKU-{sku_n:03d}",
                "Category"     : cat,
                "Item"         : item,
                "Variant"      : variant,
                "Selling_Price": float(price),
                "Cost_Price"   : round(float(price) * COST_RATIO[cat], 2),
                "Current_Stock": random.randint(safety, safety + 200),
                "Safety_Stock" : safety,
            })
            sku_n += 1

    return pd.DataFrame(records)

## 7. Table 3 — Flight_Schedules

- Key design : fixed flight numbers + fixed operating weekdays

    e.g. CX101 flies daily, MU502 flies Mon/Tue/Thu/Fri/Sun

- mirrors real-world airline schedule structure

- ±5 min random delay added to departure time for realism

In [19]:
def generate_flights(weeks: int = 26) -> pd.DataFrame:
    records   = []
    base_date = datetime(2024, 1, 1)   # 2024-01-01 = Monday

    for (fno, dest, via, airline, dep_h, dep_m, weekdays) in FLIGHT_DEF:
        for week in range(weeks):
            for wd in weekdays:
                # base_date (Monday) + week offset + weekday offset
                dep_time = (base_date
                            + timedelta(weeks=week, days=wd,
                                        hours=dep_h, minutes=dep_m)
                            + timedelta(minutes=random.randint(-5, 5)))  # ±5 min random delay
                records.append({
                    "Flight_No"     : fno,                        # ← fixed flight number
                    "Airline"       : airline,
                    "Destination"   : dest,
                    "Via"           : via,
                    "Departure_Time": dep_time.strftime("%Y-%m-%d %H:%M:%S"),
                    "Terminal_Gate" : f"T1-{random.randint(1, 55):02d}",
                    "Expected_Pax"  : random.randint(150, 400),
                })

    df = pd.DataFrame(records)
    df = df.sort_values("Departure_Time").reset_index(drop=True)
    return df

## 8. Table 4 — Holiday_Events  (Jan–Jun, 9 holiday events)

- Purpose: define holiday periods and their target category.
- No price discounts — the shop does not run formal promotions.
- Instead, purchase patterns shift during holidays:
    travellers buy more of certain items around key events
    (e.g. more Honey around Lunar New Year, 
    more Confectionery around Easter).

Columns:
- Event_ID         : unique identifier
- Event_Name       : holiday or seasonal event name
- Start_Date       : start of holiday window
- End_Date         : end of holiday window
- Target_Category  : category with elevated purchase probability
- Boost_Weight     : extra weight added to target category during the window (on top of PREF_MAP)

In [20]:
def generate_holiday_events() -> pd.DataFrame:
    # (name, start, end, target_category, boost_weight)
    # Format: (name, start, end, target_category, boost_weight,
    #           nat_boost — comma-separated nationalities with elevated visit share,
    #           nat_weight — corresponding weights during the period)
    # nat_boost/nat_weight = None means no nationality shift for that holiday event
    holiday_events = [
        # ── January ─────────────────────────────────────────────
        # New Year: international travel peaks around this period
        ("New Year Kickoff",    "2024-01-01", "2024-01-07", "Souvenir",      20,
         None, None),
        # ── February ────────────────────────────────────────────
        # Lunar New Year: CN/KR visitor surge — major outbound travel period
        ("Lunar New Year",      "2024-02-08", "2024-02-18", "Honey",         30,
         "CN,KR", "0.38,0.14"),
        # ── March / April ───────────────────────────────────────
        # Easter Long Weekend: AU/GB visitor surge — major holiday travel period
        ("Easter Long Weekend", "2024-03-29", "2024-04-01", "Confectionery", 20,
         "AU,GB", "0.35,0.14"),
    ]

    return pd.DataFrame([
        {
            "Event_ID"         : f"E-{i:02d}",
            "Event_Name"       : name,
            "Start_Date"       : start,
            "End_Date"         : end,
            "Target_Category"  : cat,
            "Boost_Weight"     : boost,
            "Nat_Boost"        : nat_boost,   # e.g. "CN,KR" or None
            "Nat_Weight"       : nat_weight,  # e.g. "0.38,0.14" or None
        }
        for i, (name, start, end, cat, boost, nat_boost, nat_weight)
        in enumerate(holiday_events, 1)
    ])

## 9. Table 5 — Sales_Transactions  (5,000 records)


Key improvement : transaction time = 30 min–3 hrs before departure
1. Collect all flights departing on the chosen date
2. Pick one flight, then calculate purchase time within pre-departure window
3. Purchase time must fall within operating hours (06:00–23:00)

In [ ]:
def generate_transactions(
    customers_df  : pd.DataFrame,
    products_df   : pd.DataFrame,
    flights_df    : pd.DataFrame,
    holiday_events_df : pd.DataFrame,
    n             : int = 5000,
) -> pd.DataFrame:

    n = int(n)
    
    # ── lookup preparation ────────────────────────────────────
    cust_df  = customers_df.set_index("Customer_ID")
    sku_df   = products_df.set_index("Product_SKU")
    cat_skus = {cat: products_df.loc[products_df["Category"]==cat,"Product_SKU"].tolist()
                for cat in CATEGORIES}

    # group flights by date for fast lookup
    flights_df["_date"]   = pd.to_datetime(flights_df["Departure_Time"]).dt.date
    flights_df["_dep_dt"] = pd.to_datetime(flights_df["Departure_Time"])
    date_to_flights = (
        flights_df.groupby("_date")
                  .apply(lambda g: g[["Flight_No","Destination","_dep_dt"]].to_dict("records"),
                         include_groups=False)
                  .to_dict()
    )
    all_dates = sorted(date_to_flights.keys())

    # Holiday event lookup
    holiday_lookup = []
    for _, r in holiday_events_df.iterrows():
        entry = {
            "Event_ID"  : r["Event_ID"],
            "Event_Name": r["Event_Name"],
            "Start"     : pd.to_datetime(r["Start_Date"]),
            "End"       : pd.to_datetime(r["End_Date"]),
            "Cat"       : r["Target_Category"],
            "Boost"     : r["Boost_Weight"],
            "Nat_Boost" : None,
        }
        if pd.notna(r["Nat_Boost"]) and pd.notna(r["Nat_Weight"]):
            nats    = r["Nat_Boost"].split(",")
            weights = [float(w) for w in r["Nat_Weight"].split(",")]
            entry["Nat_Boost"] = dict(zip(nats, weights))
        holiday_lookup.append(entry)

    cust_ids = customers_df["Customer_ID"].tolist()
    pay_map = { # Credit/Debit Card, Cash, AliPay, WeChat Pay, Digital Wallet   
        "AU": [50, 10,  0,  0, 40], "CN": [10, 18, 38, 18, 16],
        "NZ": [52, 10,  0,  0, 38], "US": [48,  8,  0,  0, 44],
        "GB": [55, 15,  0,  0, 30], "KR": [46, 18,  1,  0, 35],
        "IN": [40, 38,  0,  0, 22], "JP": [32, 30,  0,  0, 38],
        "PH": [40, 38,  0,  0, 22], "CA": [48,  8,  0,  0, 44],
        "SG": [45, 10,  2,  1, 42], "TH": [38, 32,  0,  0, 30],
    }

    # ── helper: select one SKU and calculate line amounts ────
    def pick_sku_and_amounts(nat, category, sku_pool):
        """Select SKU with CN premium preference; return (sku, unit_price, qty, amt, cost, promo_id)"""

        # Check item-level promo first
        matched_promo = None
        cat_promos = [p for p in ITEM_PROMOS if p["category"] == category]
        for ip in cat_promos:
            if random.random() < float(PROMO_TRIGGER_RATE[ip["promo_id"]]):
                matched_promo = ip
                break

        if matched_promo is None:
            # CN premium SKU preference
            if nat == "CN" and category == "Honey" and random.random() < CN_PREMIUM_TRIGGER_RATE:
                pool = [s for s in sku_pool if sku_df.loc[s, "Variant"] in CN_PREMIUM_HONEY_VARIANTS]
                sku  = random.choice(pool) if pool else random.choice(sku_pool)
            elif nat == "CN" and category == "Liquor" and random.random() < CN_PREMIUM_TRIGGER_RATE:
                pool = [s for s in sku_pool if sku_df.loc[s, "Selling_Price"] >= CN_PREMIUM_LIQUOR_MIN_PRICE]
                sku  = random.choice(pool) if pool else random.choice(sku_pool)
            else:
                sku = random.choice(sku_pool)

            prod       = sku_df.loc[sku]
            unit_price = prod["Selling_Price"]
            qty        = random.choices([1,2,3,4,5], weights=[20,12,25,35,8])[0] if nat == "CN"                          else random.choices([1,2,3,4,5], weights=[50,25,12,8,5])[0]
            line_amt   = round(unit_price * qty, 2)
            line_cost  = round(prod["Cost_Price"] * qty, 2)
            return sku, unit_price, qty, line_amt, line_cost, None

        else:
            ip = matched_promo
            if ip["item_filter"]:
                eligible = [s for s in sku_pool if sku_df.loc[s, "Item"] == ip["item_filter"]]
                if ip.get("variant_filter"):
                    eligible = [s for s in eligible if sku_df.loc[s, "Variant"] == ip["variant_filter"]]
            else:
                eligible = sku_pool

            if not eligible:
                sku        = random.choice(sku_pool)
                prod       = sku_df.loc[sku]
                unit_price = prod["Selling_Price"]
                qty        = random.choices([1,2,3,4,5], weights=[30,25,12,28,5])[0]
                return sku, unit_price, qty, round(unit_price*qty,2), round(prod["Cost_Price"]*qty,2), None

            sku        = random.choice(eligible)
            prod       = sku_df.loc[sku]
            unit_price = prod["Selling_Price"]

            if ip["promo_type"] == "BUY_X_GET_Y_ITEM":
                qty       = ip["buy_qty"] + ip["free_qty"]
                paid_qty  = ip["buy_qty"]
                line_amt  = round(unit_price * paid_qty, 2)
                line_cost = round(prod["Cost_Price"] * qty, 2)
            elif ip["promo_type"] == "BUNDLE_FIXED_ITEM":
                qty       = ip["buy_qty"]
                line_amt  = round(ip["bundle_price"], 2)
                line_cost = round(prod["Cost_Price"] * qty, 2)
            elif ip["promo_type"] == "MIX_BUY_X_GET_Y":
                qty       = ip["buy_qty"] + ip["free_qty"]
                paid_qty  = ip["buy_qty"]
                line_amt  = round(unit_price * paid_qty, 2)
                line_cost = round(prod["Cost_Price"] * paid_qty + ip.get("free_item_cost", 0), 2)                             if ip.get("free_item_cost") is not None                             else round(prod["Cost_Price"] * qty, 2)
            else:
                qty = 1; line_amt = round(unit_price, 2); line_cost = round(prod["Cost_Price"], 2)

            return sku, unit_price, qty, line_amt, line_cost, ip["promo_id"]

    # ── main loop ─────────────────────────────────────────────
    tx_records   = []   # Sales_Transactions header rows
    item_records = []   # Transaction_Items line rows
    attempts     = 0

    while len(tx_records) < n and attempts < n * 10:
        attempts += 1

        # ① pick a random date
        date = random.choice(all_dates)
        day_flights = date_to_flights.get(date, [])
        if not day_flights:
            continue

        # ② pick flight
        flight   = random.choice(day_flights)
        dep_dt   = flight["_dep_dt"]

        # ③ purchase time 30–180 min before departure
        minutes_before = random.randint(30, 180)
        trans_dt = dep_dt - timedelta(minutes=minutes_before)
        if not (6 <= trans_dt.hour < 23):
            continue

        # ④ check holiday window
        active_holiday = None
        for h in holiday_lookup:
            if h["Start"].date() <= trans_dt.date() <= h["End"].date():
                active_holiday = h
                break

        # ⑤ pick customer (with nationality boost for holidays)
        if active_holiday and active_holiday["Nat_Boost"]:
            nat_boost_map = active_holiday["Nat_Boost"]
            adj_weights   = [nat_boost_map.get(nat, w) for nat, w in zip(NATIONALITIES, NATIONALITY_WEIGHT)]
            total_w       = sum(adj_weights)
            adj_weights   = [w / total_w for w in adj_weights]
            nat           = random.choices(NATIONALITIES, weights=adj_weights)[0]
            nat_customers = customers_df.loc[customers_df["Nationality"] == nat, "Customer_ID"].tolist()
            cust_id       = random.choice(nat_customers) if nat_customers else random.choice(cust_ids)
            nat           = cust_df.loc[cust_id, "Nationality"]
        else:
            cust_id = random.choice(cust_ids)
            nat     = cust_df.loc[cust_id, "Nationality"]
        dest = flight["Destination"]

        # ⑥ decide how many items in this transaction
        # CN: gift-buying culture → more items per transaction
        if nat == "CN":
            n_items = random.choices([1,2,3,4], weights=[25,35,25,15])[0]
        else:
            n_items = random.choices([1,2,3,4], weights=[50,30,15,5])[0]

        # ⑦ generate line items — each item picks its own category
        trans_id   = f"T-{len(tx_records)+1:05d}"
        event_id   = active_holiday["Event_ID"]   if active_holiday else None
        event_name = active_holiday["Event_Name"] if active_holiday else None

        lines         = []
        total_amt_sum = 0.0
        total_cost_sum= 0.0

        for line_no in range(1, n_items + 1):
            category = pick_category(nat, dest, active_holiday)
            sku_pool = cat_skus.get(category, products_df["Product_SKU"].tolist())
            sku, unit_price, qty, line_amt, line_cost, promo_id =                 pick_sku_and_amounts(nat, category, sku_pool)

            lines.append({
                "Trans_ID"    : trans_id,
                "Line_No"     : line_no,
                "Product_SKU" : sku,
                "Category"    : category,
                "Quantity"    : qty,
                "Unit_Price"  : unit_price,
                "Line_Amount" : line_amt,
                "Line_Cost"   : line_cost,
                "Gross_Profit": round(line_amt - line_cost, 2),
                "Promo_ID"    : promo_id,
            })
            total_amt_sum  += line_amt
            total_cost_sum += line_cost

        # ⑧ build transaction header
        tx_records.append({
            "Trans_ID"      : trans_id,
            "Date_Time"     : trans_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "Customer_ID"   : cust_id,
            "Flight_No"     : flight["Flight_No"],
            "Destination"   : dest,
            "Event_ID"      : event_id,
            "Event_Name"    : event_name,
            "Item_Count"    : n_items,
            "Total_Amount"  : round(total_amt_sum, 2),
            "Total_Cost"    : round(total_cost_sum, 2),
            "Gross_Profit"  : round(total_amt_sum - total_cost_sum, 2),
            "Payment_Method": random.choices(
                PAYMENT_METHODS,
                weights=pay_map.get(nat, [45,20,5,5,25])
            )[0],
        })
        item_records.extend(lines)

    return pd.DataFrame(tx_records), pd.DataFrame(item_records)

## 10. SAVE TO CSV + LOAD INTO SQLITE


In [22]:
def save_to_csv(datasets: dict) -> None:
    for name, df in datasets.items():
        path = os.path.join(OUTPUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"  ✅ {name}.csv  →  {len(df):>5,} rows × {len(df.columns):>2} columns")

In [23]:
def save_to_sqlite(datasets: dict, db_path: str = "duty_free.db") -> None:
    conn = sqlite3.connect(db_path)
    for name, df in datasets.items():
        # drop internal helper columns before saving
        save_df = df.drop(columns=[c for c in ["_date","_dep_dt"] if c in df.columns])
        save_df.to_sql(name, conn, if_exists="replace", index=False)
    conn.close()
    print(f"\n  🗄️  SQLite DB created  →  {db_path}")

## 11. VALIDATION SUMMARY


In [26]:
def print_summary(datasets: dict) -> None:
    tx   = datasets["Sales_Transactions"]
    ti   = datasets["Transaction_Items"]
    cp   = datasets["Customer_Profiles"]

    print("\n" + "="*58)
    print("  📊 Validation Summary")
    print("="*58)

    print("\n  [Transaction share by nationality]")
    merged = tx.merge(cp[["Customer_ID","Nationality"]], on="Customer_ID")
    nat_pct = merged["Nationality"].value_counts(normalize=True).mul(100).round(1)
    for k, v in nat_pct.items():
        print(f"    {k} : {v}%")

    print("\n  [Avg items per transaction by nationality]")
    avg_items = merged.groupby("Nationality")["Item_Count"].mean().round(2).sort_values(ascending=False)
    for k, v in avg_items.items():
        print(f"    {k} : {v} items")

    print("\n  [Revenue share & avg margin by category]")
    grp = ti.groupby("Category").agg(
        Revenue=("Line_Amount","sum"),
        Cost=("Line_Cost","sum")
    )
    grp["Rev%"]    = (grp["Revenue"] / grp["Revenue"].sum() * 100).round(1)
    grp["Margin%"] = ((grp["Revenue"] - grp["Cost"]) / grp["Revenue"] * 100).round(1)
    for cat, row in grp.sort_values("Rev%", ascending=False).iterrows():
        print(f"    {cat:15s}: revenue {row['Rev%']:>5}%  margin {row['Margin%']:>5}%")

    print("\n  [Average transaction value (ATV) by destination]")
    atv = tx.groupby("Destination")["Total_Amount"].mean().round(2).sort_values(ascending=False)
    for k, v in atv.items():
        print(f"    {k:15s}: AUD {v}")

    print("\n  [ATV by nationality]")
    atv_nat = merged.groupby("Nationality")["Total_Amount"].mean().round(2).sort_values(ascending=False)
    for k, v in atv_nat.items():
        print(f"    {k} : AUD {v}")

    print("\n  [Transaction vs departure time gap validation]")
    fl = datasets["Flight_Schedules"][["Flight_No","Departure_Time"]].copy()
    fl["Departure_Time"] = pd.to_datetime(fl["Departure_Time"])
    fl["_dep_date"] = fl["Departure_Time"].dt.date
    tx_copy = tx.assign(_trans_date=pd.to_datetime(tx["Date_Time"]).dt.date)
    chk = tx_copy.merge(fl, left_on=["Flight_No","_trans_date"],
                            right_on=["Flight_No","_dep_date"], how="left")
    chk["mins_before"] = (
        (chk["Departure_Time"] - pd.to_datetime(chk["Date_Time"]))
        .dt.total_seconds() / 60
    )
    valid = chk["mins_before"].between(30, 180)
    print(f"    Purchases within 30–180 min before departure : {valid.mean()*100:.1f}%")
    print(f"    Avg purchase timing : {chk['mins_before'].mean():.0f} min before departure")

    print("\n  [Holiday event coverage]")
    print(f"    Transactions during holiday : {tx['Event_ID'].notna().mean()*100:.1f}%")
    if tx["Event_ID"].notna().any():
        event_counts = tx[tx["Event_ID"].notna()]["Event_Name"].value_counts()
        for evt, cnt in event_counts.items():
            print(f"    {evt:30s}: {cnt:,} transactions")
    print("="*58)

## 12. MAIN

In [29]:
if __name__ == "__main__":
    print("="*58)
    print("  Sydney T1 Duty-Free — Dataset Generator")
    print("="*58 + "\n")
    print("⏳ Generating data...\n")

    customers                      = generate_customers(4000)
    products                       = generate_products()
    flights                        = generate_flights(weeks=26)
    holiday_events                 = generate_holiday_events()
    transactions, transaction_items = generate_transactions(
        customers, products, flights, holiday_events, n=5000
    )

    flights_clean = flights.drop(columns=["_date","_dep_dt"], errors="ignore")

    datasets = {
        "Customer_Profiles"  : customers,
        "Product_Inventory"  : products,
        "Flight_Schedules"   : flights_clean,
        "Holiday_Events"     : holiday_events,
        "Sales_Transactions" : transactions,
        "Transaction_Items"  : transaction_items,
    }

    print("📁 Saving CSVs...\n")
    save_to_csv(datasets)

    save_to_sqlite({**datasets, "Flight_Schedules": flights}, db_path="duty_free.db")

    print_summary({
        "Sales_Transactions": transactions,
        "Transaction_Items" : transaction_items,
        "Customer_Profiles" : customers,
        "Flight_Schedules"  : flights,
    })

    print(f"\n🎉 Done! Output folder : ./{OUTPUT_DIR}/")
    print("   duty_free.db      : SQLite DB for SQL practice")

  Sydney T1 Duty-Free — Dataset Generator

⏳ Generating data...

📁 Saving CSVs...

  ✅ Customer_Profiles.csv  →  4,000 rows ×  6 columns
  ✅ Product_Inventory.csv  →    108 rows ×  8 columns
  ✅ Flight_Schedules.csv  →  2,860 rows ×  7 columns
  ✅ Holiday_Events.csv  →      3 rows ×  8 columns
  ✅ Sales_Transactions.csv  →  5,000 rows × 12 columns
  ✅ Transaction_Items.csv  →  9,365 rows × 10 columns

  🗄️  SQLite DB created  →  duty_free.db

  📊 Validation Summary

  [Transaction share by nationality]
    CN : 21.3%
    AU : 11.3%
    US : 9.4%
    GB : 9.0%
    KR : 8.3%
    IN : 7.8%
    JP : 7.8%
    NZ : 7.7%
    PH : 5.0%
    TH : 4.3%
    CA : 4.3%
    SG : 3.7%

  [Avg items per transaction by nationality]
    CN : 2.34 items
    GB : 1.82 items
    KR : 1.82 items
    TH : 1.76 items
    AU : 1.75 items
    PH : 1.75 items
    IN : 1.74 items
    JP : 1.74 items
    SG : 1.74 items
    NZ : 1.7 items
    CA : 1.68 items
    US : 1.68 items

  [Revenue share & avg margin by cat